<font size="10">Machine Learning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import xgboost as xgb
import lightgbm as lgb

In [ ]:
# 基础设置
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
matplotlib.rcParams['font.family'] = 'Times New Roman'
matplotlib.rcParams['axes.unicode_minus'] = False
os.environ["LOKY_MAX_CPU_COUNT"] = "8"

In [ ]:
# 数据加载
file_name =[] 
df = pd.read_excel(file_name)

In [ ]:
# 最优特征选择
SAMCFS = []
target = []

In [ ]:
# 自定义ANN
class SimpleANN(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=None, hidden_dim=64, activation='relu', lr=0.02, batch_size=64, epochs=300, device='cpu'):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.activation = activation
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = device
        self.model = None

    def get_activation(self):
        if self.activation == 'relu':
            return nn.ReLU()
        elif self.activation == 'tanh':
            return nn.Tanh()
        elif self.activation == 'sigmoid':
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")

    def build_model(self):
        act = self.get_activation()
        model = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dim),
            act,
            nn.Linear(self.hidden_dim, 1)
        )
        return model.to(self.device)

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        dataset = TensorDataset(X, y)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model = self.build_model()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        self.model.train()
        for epoch in range(self.epochs):
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                output = self.model(batch_x).squeeze()
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
        return self

    def predict(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            preds = self.model(X_tensor).cpu().numpy()
        return preds

In [ ]:
# 绘图函数
def plot_true_vs_pred(test_true, test_pred_inv, model_name):
    plt.figure(figsize=(6, 6), dpi=200)
    plt.scatter(test_true, test_pred_inv, color="#3B86B9", label='Prediction', s=40)

    lr = LinearRegression().fit(test_true, test_pred_inv)
    plt.plot(test_true, lr.predict(test_true), color="red", linestyle="-", linewidth=1, label='Fitted Line')

    plt.title(f"{model_name} - Test Set", fontsize=14)
    plt.xlabel("True Values", fontsize=12)
    plt.ylabel("Predicted Values", fontsize=12)
    plt.legend(loc='upper left')

    test_mse = metrics.mean_squared_error(test_true, test_pred_inv)
    test_rmse = np.sqrt(test_mse)
    test_mae = metrics.mean_absolute_error(test_true, test_pred_inv)
    test_mape = np.mean(np.abs((test_true - test_pred_inv) / test_true)) * 100
    test_r2 = metrics.r2_score(test_true, test_pred_inv)

    metrics_text = (
        f"MSE={test_mse:.4f}\n"
        f"RMSE={test_rmse:.4f}\n"
        f"MAE={test_mae:.4f}\n"
        f"MAPE={test_mape:.2f}%\n"
        f"R²={test_r2:.4f}"
    )
    plt.text(0.95, 0.05, metrics_text, transform=plt.gca().transAxes,
             fontsize=10, verticalalignment='bottom', horizontalalignment='right')

    plt.grid(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# 主程序
x = df[SAMCFS]
y = df[target]

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
x_scaled = scaler_x.fit_transform(x)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))
x_scaled = pd.DataFrame(x_scaled, columns=x.columns)

train_x, test_x, train_y, test_y = train_test_split(
    x_scaled, y_scaled,
    test_size=0.2,
    random_state=25,
    shuffle=True
)

In [ ]:
# 1. Decision Tree
def run_decision_tree(train_x, train_y, test_x, test_y, scaler_y):
    model = DecisionTreeRegressor(max_depth=5, min_samples_leaf=3, min_samples_split=7, random_state=25)
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    # 反归一化
    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="Decision Tree")


# 2. Lasso
def run_lasso(train_x, train_y, test_x, test_y, scaler_y):
    model = Lasso(alpha=0.001)
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="Lasso")


# 3. SVR
def run_svr(train_x, train_y, test_x, test_y, scaler_y):
    model = SVR(C=1, epsilon=0.01, kernel='rbf')
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="SVR")


# 4. Random Forest
def run_random_forest(train_x, train_y, test_x, test_y, scaler_y):
    model = RandomForestRegressor(max_depth=5, min_samples_leaf=5, min_samples_split=2, n_estimators=150, random_state=20)
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="Random Forest")


# 5. XGBoost
def run_xgboost(train_x, train_y, test_x, test_y, scaler_y):
    model = xgb.XGBRegressor(
        colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=100,
        subsample=1.0, objective="reg:squarederror", random_state=25, enable_categorical=False
    )
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="XGBoost")


# 6. LightGBM
def run_lightgbm(train_x, train_y, test_x, test_y, scaler_y):
    model = lgb.LGBMRegressor(
        bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, lambda_l1=0.0,
        lambda_l2=0.01, learning_rate=0.05, max_depth=3, min_data_in_leaf=10,
        n_estimators=150, num_leaves=10, verbosity=-1, objective='regression', random_state=25
    )
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="LightGBM")


# 7. AdaBoost
def run_adaboost(train_x, train_y, test_x, test_y, scaler_y):
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=13, min_samples_leaf=1, min_samples_split=9, random_state=25),
        learning_rate=0.05, loss='square', n_estimators=100, random_state=25
    )
    model.fit(train_x, train_y.ravel())
    test_pred = model.predict(test_x)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="AdaBoost")


# 8. ANN
def run_ann(train_x, train_y, test_x, test_y, scaler_y):
    model = SimpleANN(
        input_dim=train_x.shape[1], hidden_dim=64, activation='relu',
        lr=0.02, batch_size=64, epochs=300, device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    model.fit(train_x.values, train_y.ravel())
    test_pred = model.predict(test_x.values)

    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    plot_true_vs_pred(test_y_inv, test_pred_inv, model_name="ANN")

In [ ]:
def plot_all_models(train_x, train_y, test_x, test_y, scaler_y):

    model_functions = [
        run_decision_tree, run_lasso, run_svr, run_random_forest,
        run_xgboost, run_lightgbm, run_adaboost, run_ann
    ]

    model_names = [
        "DT", "Lasso", "SVR", "RF",
        "XG", "LB", "AB", "ANN"
    ]

    # 子图标签
    subplot_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)']

    fig, axs = plt.subplots(4, 2, figsize=(8, 16), dpi=300)
    axs = axs.flatten()

    for i, model_name in enumerate(model_names):

        model = None

        if model_name == "DT":
            model = DecisionTreeRegressor(
                max_depth=5,
                min_samples_leaf=3,
                min_samples_split=7,
                random_state=25
            )
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "Lasso":
            model = Lasso(alpha=0.001)
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "SVR":
            model = SVR(C=1, epsilon=0.01, kernel='rbf')
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "RF":
            model = RandomForestRegressor(
                max_depth=5,
                min_samples_leaf=5,
                min_samples_split=2,
                n_estimators=150,
                random_state=20
            )
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "XG":
            model = xgb.XGBRegressor(
                colsample_bytree=1.0,
                learning_rate=0.05,
                max_depth=3,
                n_estimators=100,
                subsample=1.0,
                objective="reg:squarederror",
                random_state=25,
                enable_categorical=False
            )
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "LB":
            model = lgb.LGBMRegressor(
                bagging_fraction=0.8,
                bagging_freq=1,
                feature_fraction=0.8,
                lambda_l1=0.0,
                lambda_l2=0.01,
                learning_rate=0.05,
                max_depth=3,
                min_data_in_leaf=10,
                n_estimators=150,
                num_leaves=10,
                verbosity=-1,
                objective='regression',
                random_state=25
            )
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "AB":
            model = AdaBoostRegressor(
                estimator=DecisionTreeRegressor(
                    max_depth=13,
                    min_samples_leaf=1,
                    min_samples_split=9,
                    random_state=25
                ),
                learning_rate=0.05,
                loss='square',
                n_estimators=100,
                random_state=25
            )
            model.fit(train_x, train_y.ravel())
            test_pred = model.predict(test_x)

        elif model_name == "ANN":

            seed = 50
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = SimpleANN(
                input_dim=train_x.shape[1],
                hidden_dim=64,
                activation='relu',
                lr=0.02,
                batch_size=64,
                epochs=300,
                device='cuda' if torch.cuda.is_available() else 'cpu'
            )

            model.fit(train_x.values, train_y.ravel())
            test_pred = model.predict(test_x.values)

        # 反归一化
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
        test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

        ax = axs[i]

        # 散点图
        ax.scatter(
            test_y_inv,
            test_pred_inv,
            color="#3B86B9",
            s=30,
            label='Prediction'
        )

        # 拟合线
        lr_line = LinearRegression().fit(test_y_inv, test_pred_inv)
        ax.plot(
            test_y_inv,
            lr_line.predict(test_y_inv),
            color="red",
            linestyle="-",
            linewidth=1,
            label='Fitted Line'
        )

        # 坐标轴标签
        ax.set_xlabel("True Values", fontsize=10)
        ax.set_ylabel("Predicted Values", fontsize=10)

        # 图例
        ax.legend(
            loc='upper left',
            fontsize=9,
            frameon=False
        )

        # 指标计算
        mse = metrics.mean_squared_error(test_y_inv, test_pred_inv)
        rmse = np.sqrt(mse)
        mae = metrics.mean_absolute_error(test_y_inv, test_pred_inv)
        mape = np.mean(np.abs((test_y_inv - test_pred_inv) / test_y_inv)) * 100
        r2 = metrics.r2_score(test_y_inv, test_pred_inv)

        metrics_text = (
            f"MSE={mse:.4f}\n"
            f"RMSE={rmse:.4f}\n"
            f"MAE={mae:.4f}\n"
            f"MAPE={mape:.2f}%\n"
            f"R²={r2:.4f}"
        )

        ax.text(
            0.95,
            0.05,
            metrics_text,
            transform=ax.transAxes,
            fontsize=8,
            verticalalignment='bottom',
            horizontalalignment='right'
        )

        # 子图标签 (a)-(h) 放在下方
        ax.text(
            0.5,
            -0.15,
            f"{subplot_labels[i]} {model_name}",
            transform=ax.transAxes,
            ha='center',
            va='top',
            fontsize=12
        )

        ax.grid(False)

    plt.tight_layout(rect=[0, 0.02, 1, 1])
    plt.savefig(
        '8models_final.svg',
        format='svg',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

In [ ]:
plot_all_models(train_x, train_y, test_x, test_y, scaler_y)

<font size="10">SHAP分布图

In [ ]:
import shap

In [ ]:
class SHAPVisualizer:
    def __init__(self, model, model_name, feature_names):
        self.model = model
        self.model_name = model_name
        self.feature_names = feature_names
        self.explainer = None

    def create_explainer(self, x_sample):
        try:
            # Tree-based models
            tree_classes = (
                lgb.LGBMRegressor,
            )
            if isinstance(self.model, tree_classes):
                self.explainer = shap.TreeExplainer(self.model)
            else:
                background = shap.kmeans(x_sample, 10)
                self.explainer = shap.KernelExplainer(self.model.predict, background)
        except Exception as e:
            print(f"[❌ ERROR] 创建 explainer 失败: {e}")

    def plot_summary(self, x_sample):
        print(f"\n📊 SHAP Summary Plot for: {self.model_name}")
        try:
            if self.explainer is None:
                self.create_explainer(x_sample)
            shap_values = self.explainer.shap_values(x_sample)

            # 先画shap summary图，但不show
            shap.summary_plot(
                shap_values, 
                features=x_sample, 
                feature_names=self.feature_names, 
                show=False
            )

            # 修改画布大小
            plt.gcf().set_size_inches(7, 4)  # 这里可以调整成你需要的，比如10×8英寸

            # 修改字体大小
            plt.xticks(fontsize=9)
            plt.yticks(fontsize=9)
            plt.xlabel("SHAP value (impact on model output)", fontsize=10)
           # plt.ylabel("Features", fontsize=10)
             # =========== 重点：修改右侧Colorbar字体 ==============
            colorbar = plt.gcf().axes[-1]   # Colorbar一般是最后一个ax
            colorbar.tick_params(labelsize=9)  # 刻度（High / Low）字体大小
            colorbar.set_ylabel("Feature value", fontsize=9)  # Colorbar主标题字体

            # 保存高分辨率图片
            #plt.savefig("shap_summary_lightgbm.svg", format='svg', dpi=300, bbox_inches='tight')

            plt.show()

        except Exception as e:
            print(f"[❌ ERROR] SHAP summary 绘制失败: {e}")

In [ ]:
# ================= 训练最优 LightGBM模型 ==================
best_model = lgb.LGBMRegressor(
    bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8,
    lambda_l1=0.0, lambda_l2=0.01, learning_rate=0.05, max_depth=3,
    min_data_in_leaf=10, n_estimators=150, num_leaves=10,
    verbosity=-1, objective='regression', random_state=25
)
best_model.fit(train_x, train_y.ravel())

In [ ]:
model = best_model

In [ ]:
feature_importance = model.feature_importances_ 
feature_names = x.columns 
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=True)

plt.figure(figsize=(9, 4),dpi=300) 
colors_fi = plt.cm.viridis(np.linspace(0, 1, len(importance_df))) 
bars = plt.barh(range(len(importance_df)), importance_df['importance'], color=colors_fi, align='center')
plt.yticks(range(len(importance_df)), importance_df['feature'], fontname='Times New Roman') 
plt.xlabel('Feature Importance', fontsize=12, fontweight='black', fontname='Times New Roman')
#plt.ylabel('Features', fontsize=12, fontweight='bold', fontname='Times New Roman')
plt.title('', fontsize=18, fontweight='bold', fontname='Times New Roman')
plt.gca().spines['top'].set_visible(False) 
plt.gca().spines['right'].set_visible(False)
plt.grid(axis='x', alpha=0.3, linestyle='--') 

for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width + max(importance_df['importance']) * 0.01, 
             bar.get_y() + bar.get_height()/2, 
             f'{width:.0f}', 
             ha='left', va='center', fontweight='normal', fontname='Times New Roman')

plt.tight_layout() 
plt.show()

In [ ]:
shap_vis = SHAPVisualizer(
    model=best_model, 
    model_name="LightGBM", 
    feature_names=train_x.columns.tolist()
)

In [ ]:
shap_vis.plot_summary(train_x)

<font size="10">SHAP Top 10 

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import lightgbm as lgb

class SHAPVisualizer:
    def __init__(self, model, model_name, feature_names):
        self.model = model
        self.model_name = model_name
        self.feature_names = feature_names
        self.explainer = None

    def create_explainer(self, x_sample):
        try:
            tree_classes = (lgb.LGBMRegressor,)
            if isinstance(self.model, tree_classes):
                self.explainer = shap.TreeExplainer(self.model)
            else:
                background = shap.kmeans(x_sample, 10)
                self.explainer = shap.KernelExplainer(self.model.predict, background)
        except Exception as e:
            print(f"[❌ ERROR] 创建 explainer 失败: {e}")

    def plot_summary(self, x_sample):
        print(f"\n📊 SHAP Summary Plot for: {self.model_name}")
        try:
            if self.explainer is None:
                self.create_explainer(x_sample)
            shap_values = self.explainer.shap_values(x_sample)
            shap.summary_plot(shap_values, features=x_sample, feature_names=self.feature_names, show=True)
        except Exception as e:
            print(f"[❌ ERROR] SHAP summary 绘制失败: {e}")

    def plot_top10_features(self, x_sample):
        print(f"\n📊 Top 10 Feature Importance for: {self.model_name}")
        try:
            if self.explainer is None:
                self.create_explainer(x_sample)
            shap_values = self.explainer.shap_values(x_sample)

            # 计算每个特征的平均绝对SHAP值
            shap_importance = pd.DataFrame({
                'Feature': self.feature_names,
                'Importance': [np.mean(np.abs(shap_values[:, i])) for i in range(len(self.feature_names))]
            })

            # 排序选取Top 10特征，并反向（最重要在最上面）
            top10_shap = shap_importance.sort_values(by='Importance', ascending=False).head(10)
            top10_shap = top10_shap[::-1]  # 反向，让重要特征在最上面

            # 绘制Top 10特征贡献的条形图
            plt.figure(figsize=(9, 4), dpi=300)
            bars = plt.barh(
                top10_shap['Feature'],
                top10_shap['Importance'],
                color=plt.cm.Blues(np.linspace(0.4, 1, 10))  # 蓝色渐变
            )
            plt.xlabel(r'mean(|SHAP value|) (average impact on model output magnitude)', fontsize=10)
            plt.xticks(fontsize=9)
            plt.yticks(fontsize=9)
            plt.tight_layout()

            # 在每个条形图的右侧添加数值
            for bar in bars:
                # 获取条形的宽度（SHAP值）
                width = bar.get_width()
                # 在条形图的右边紧跟着加上文本（数值）
                plt.text(width + 0.0001, bar.get_y() + bar.get_height() / 2,
                         f'{width:.4f}', ha='left', va='center', fontsize=8, color='black'
            plt.show()

        except Exception as e:
            print(f"[❌ ERROR] SHAP Top 10 feature importance 绘制失败: {e}")

In [ ]:
shap_vis = SHAPVisualizer(
    model=best_model,
    model_name="LightGBM",
    feature_names=train_x.columns.tolist()
)

In [ ]:
shap_vis.plot_top10_features(train_x)

<font size="10">PDP

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(test_x)
mean_shap_values = np.abs(shap_values).mean(axis=0)

In [ ]:
shap_df = pd.DataFrame({'feature': test_x.columns,'mean_shap': mean_shap_values  })
top8 = shap_df.sort_values(by='mean_shap', ascending=False).head(8)['feature'].tolist()
fig, axes = plt.subplots(4, 2, figsize=(12, 20))  # 4 rows, 2 columns
#fig.suptitle('Partial Dependence Plots - Top 8 Features', fontsize=15,
             #fontweight='normal', fontname='Times New Roman', y=0.95)

for idx, feat in enumerate(top8):
    ax = axes[idx//2, idx%2]  # Adjusted for 4 rows and 2 columns
    disp = PartialDependenceDisplay.from_estimator(
        best_model, test_x, [feat], kind='average', grid_resolution=50, ax=None
    )
    xvals = disp.axes_[0,0].lines[0].get_xdata()
    yvals = disp.axes_[0,0].lines[0].get_ydata()
    plt.close(disp.figure_)
    ax.plot(xvals, yvals, color=COLORS['pdp_line'], linewidth=3, zorder=3)
    ax.fill_between(xvals, yvals, alpha=0.3, color=COLORS['pdp_fill'], zorder=1)
    #rug = test_x[feat].values
    #ymin = ax.get_ylim()[0]
    #ax.scatter(rug, [ymin]*len(rug), alpha=0.4, s=10, color=COLORS['accent'], marker='|', zorder=2)
    corr = np.corrcoef(test_x[feat], test_y_inv)[0,1]
    #ax.text(0.05, 0.95, f'Corr: {corr:.2f}', transform=ax.transAxes,
            #fontsize=9, va='top', bbox=dict(facecolor='white', alpha=0.7))
    #ax.set_title(f'PDP: {feat}', fontsize=12, fontweight='normal', fontname='Times New Roman')
    ax.set_xlabel(feat, fontsize=13, fontweight='normal', fontname='Times New Roman')
    ax.set_ylabel('Partial Dependence', fontsize=13, fontweight='normal', fontname='Times New Roman')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    #if idx == 0:
        #ax.legend(['PDP', 'Rug'], fontsize=9, prop={'family':'Times New Roman'})

plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to fit titles

fig.savefig('partial_dependence_plots_top1_features.svg', format='svg', dpi=300)
plt.show() 

In [ ]:
plt.rcParams.update({'font.size': 13, 'font.family': 'Times New Roman'})
top_2 = shap_df.sort_values(by='mean_shap', ascending=False).head(2)['feature'].tolist()
if len(top_2) >= 2:
    print(f"生成2D PDP: {top_2[0]} vs {top_2[1]}")
    fig, ax = plt.subplots(1,1, figsize=(10,8))
    try:
        disp2 = PartialDependenceDisplay.from_estimator(
            estimator=best_model,
            X=test_x,
            features=[(top_2[0], top_2[1])],
            kind='average',
            grid_resolution=20,
            ax=ax,
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
        ax.set_xlabel(top_2[0], fontsize=13, fontweight='normal', fontname='Times New Roman')
        ax.set_ylabel(top_2[1], fontsize=13, fontweight='normal', fontname='Times New Roman')
        
        # 自定义colorbar
        if disp2.figure_ is not None and len(disp2.figure_.axes) > 1:
            cb_ax = disp2.figure_.axes[-1]
            if cb_ax is not ax:
                try:
                    if hasattr(disp2, 'colorbar') and disp2.colorbar is not None:
                        disp2.colorbar.set_label('Partial Dependence', size=13, weight='normal', family='Times New Roman')
                        for lbl in disp2.colorbar.ax.get_yticklabels(): lbl.set_fontname('Times New Roman')
                    else:
                        cb_ax.yaxis.label.set_fontsize(13)
                        cb_ax.yaxis.label.set_fontweight('normal')
                        cb_ax.yaxis.label.set_fontname('Times New Roman')
                        if cb_ax.get_ylabel() == "": cb_ax.set_ylabel('Partial Dependence', fontsize=13, fontweight='normal', fontname='Times New Roman')
                        for t in cb_ax.get_yticklabels(): t.set_fontname('Times New Roman')
                except Exception as e_cb:
                    print(f"自定义颜色条失败: {e_cb}")
        plt.tight_layout()
        fig.savefig('2D_PDP_top2_features.svg', format='svg', dpi=300)
        plt.show()
    except Exception as e:
        print(f"2D PDP生成失败: {e}")
else:
    print("至少需要两个特征来绘制2D PDP。")